# Exemplo 02 - Algoritmo Genetico: Problema da Mochila (Knapsack)

---

## O Problema

Voce tem uma mochila com capacidade limitada de peso (**7 kg**).  
Ha varios itens disponiveis, cada um com um **peso** e um **valor**.  
Objetivo: escolher a combinacao de itens que **MAXIMIZE** o valor total, sem ultrapassar o peso maximo.

## Onde isso aparece na vida real?

| Area | Exemplo |
|------|--------|
| Logistica | Carregar um caminhao com as cargas mais valiosas |
| Financas | Montar uma carteira de investimentos com orcamento limitado |
| Computacao | Alocar tarefas em servidores com memoria limitada |
| Industria | Cortar material minimizando desperdicio |

## Representacao Genetica

Cada individuo (cromossomo) e uma **lista binaria**: `[0, 1, 0, 1, 1, ...]`
- **1** = item incluido na mochila
- **0** = item excluido

## Componentes do AG que vamos usar

1. Representacao binaria (cromossomo)
2. Funcao de aptidao (fitness) com penalizacao
3. Selecao por roleta (do Exemplo 01!)
4. Crossover de um ponto (cruzamento)
5. Mutacao por inversao de bit

In [ ]:
import random

---
## Passo 1: Definir o problema

Vamos definir os **itens disponiveis** (com nome, peso e valor) e a **capacidade maxima** da mochila.

In [ ]:
# Itens disponiveis: cada item tem nome, peso (kg) e valor (R$)
itens = [
    {"nome": "Notebook",      "peso": 3.0, "valor": 2000},
    {"nome": "Livro de IA",   "peso": 1.5, "valor": 300},
    {"nome": "Garrafa agua",  "peso": 1.0, "valor": 50},
    {"nome": "Camera",        "peso": 2.0, "valor": 1500},
    {"nome": "Casaco",        "peso": 1.5, "valor": 200},
    {"nome": "Lanche",        "peso": 0.5, "valor": 80},
    {"nome": "Carregador",    "peso": 0.5, "valor": 150},
    {"nome": "Tablet",        "peso": 1.0, "valor": 1200},
]

PESO_MAXIMO = 7.0  # Capacidade maxima da mochila em kg
NUM_ITENS = len(itens)

# Mostrar os itens
print("ITENS DISPONIVEIS")
print("-" * 40)
for item in itens:
    print(" ", item["nome"], "\t", item["peso"], "kg \tR$", item["valor"])
print("-" * 40)
print("Peso maximo da mochila:", PESO_MAXIMO, "kg")
print("Quantidade de itens:", NUM_ITENS)

---
## Passo 2: Parametros do Algoritmo Genetico

Estes sao os **hiperparametros** que controlam o comportamento do AG.

| Parametro | Valor | O que faz |
|-----------|-------|----------|
| Tamanho da populacao | 20 | Quantos individuos por geracao |
| Numero de geracoes | 50 | Quantas geracoes vamos evoluir |
| Taxa de crossover | 80% | Chance de cruzamento entre pais |
| Taxa de mutacao | 5% | Chance de inversao por gene |

In [ ]:
# Parametros do AG
TAMANHO_POPULACAO = 20     # Quantos individuos por geracao
NUM_GERACOES = 50          # Quantas geracoes vamos evoluir
TAXA_CROSSOVER = 0.8       # 80% de chance de crossover (cruzamento)
TAXA_MUTACAO = 0.05        # 5% de chance de mutacao por gene (bit)

print("Parametros definidos!")
print("Populacao:", TAMANHO_POPULACAO)
print("Geracoes:", NUM_GERACOES)
print("Taxa de crossover:", TAXA_CROSSOVER)
print("Taxa de mutacao:", TAXA_MUTACAO)

---
## Passo 3: Criar a populacao inicial

Cada individuo e uma **lista binaria aleatoria** de tamanho igual ao numero de itens.

Exemplo: `[1, 0, 1, 0, 0, 1, 1, 0]` significa:
- Notebook (1=sim), Livro (0=nao), Garrafa (1=sim), Camera (0=nao), Casaco (0=nao), Lanche (1=sim), Carregador (1=sim), Tablet (0=nao)

In [ ]:
def criar_individuo():
    """Cria um cromossomo aleatorio (lista de 0s e 1s)."""
    return [random.randint(0, 1) for _ in range(NUM_ITENS)]

def criar_populacao(tamanho):
    """Cria a populacao inicial com N individuos aleatorios."""
    return [criar_individuo() for _ in range(tamanho)]

# Exemplo: criar e mostrar 3 individuos aleatorios
print("Exemplos de individuos aleatorios:")
for i in range(3):
    ind = criar_individuo()
    print("  Individuo", i+1, ":", ind)

---
## Passo 4: Funcao de Aptidao (Fitness)

A funcao de fitness avalia o quao **boa** e uma solucao:

- Se o peso total **NAO excede** o limite → `fitness = valor total dos itens`
- Se o peso total **EXCEDE** o limite → `fitness = 0` (penalizacao total)

> **Nota para discussao**: Existem outras formas de penalizacao (ex: subtrair o excesso). A penalizacao total (fitness = 0) e mais simples e funciona bem aqui.

In [ ]:
def calcular_fitness(individuo):
    """
    Avalia a qualidade de um individuo (solucao candidata).
    Retorna o valor total dos itens se o peso esta dentro do limite, senao 0.
    """
    peso_total = 0
    valor_total = 0

    for i in range(NUM_ITENS):
        if individuo[i] == 1:  # Se o item i foi incluido na mochila
            peso_total += itens[i]["peso"]
            valor_total += itens[i]["valor"]

    # Penalizacao: se excedeu o peso, a solucao e invalida
    if peso_total > PESO_MAXIMO:
        return 0  # Fitness zero - solucao descartada pela selecao natural

    return valor_total

# Teste: avaliar alguns individuos
teste1 = [1, 0, 0, 1, 0, 0, 0, 1]  # Notebook + Camera + Tablet
teste2 = [1, 1, 1, 1, 1, 1, 1, 1]  # TODOS os itens (vai exceder o peso!)

print("Teste 1:", teste1)
print("  Fitness:", calcular_fitness(teste1))
print()
print("Teste 2:", teste2, "(todos os itens)")
print("  Fitness:", calcular_fitness(teste2), "(excedeu o peso!)")

---
## Passo 5: Selecao por Roleta

Mesmo conceito do **Exemplo 01**!  
Individuos com fitness maior tem mais chance de serem escolhidos como pais.

> **Cuidado**: se todos tem fitness 0, a roleta nao funciona. Tratamos esse caso especial.

In [ ]:
def selecao_roleta(populacao, fitness_lista):
    """Seleciona um individuo usando o metodo da roleta."""
    soma_total = sum(fitness_lista)

    # Caso especial: se todos tem fitness 0, escolhe aleatoriamente
    if soma_total == 0:
        return random.choice(populacao)

    ponto_sorteio = random.uniform(0, soma_total)
    soma_atual = 0

    for individuo, fit in zip(populacao, fitness_lista):
        soma_atual += fit
        if soma_atual >= ponto_sorteio:
            return individuo

    return populacao[-1]  # Seguranca

print("Funcao selecao_roleta() definida!")

---
## Passo 6: Crossover (Cruzamento) de um ponto

Combina dois pais para gerar dois filhos.  
Escolhe um **ponto de corte** aleatorio e troca as "metades" dos cromossomos.

### Exemplo visual (ponto de corte = 3):

```
Pai 1:   [1, 0, 1 | 0, 0, 1, 1, 0]
Pai 2:   [0, 1, 0 | 1, 1, 0, 0, 1]
                   |
Filho 1: [1, 0, 1 | 1, 1, 0, 0, 1]  <- inicio do Pai 1 + final do Pai 2
Filho 2: [0, 1, 0 | 0, 0, 1, 1, 0]  <- inicio do Pai 2 + final do Pai 1
```

In [ ]:
def crossover(pai1, pai2):
    """
    Realiza crossover de um ponto entre dois pais.
    Retorna dois filhos.
    """
    # Com certa probabilidade, NAO faz crossover (filhos = copias dos pais)
    if random.random() > TAXA_CROSSOVER:
        return pai1[:], pai2[:]  # [:] cria uma copia da lista

    # Escolher um ponto de corte aleatorio (entre 1 e N-1)
    ponto_corte = random.randint(1, NUM_ITENS - 1)

    # Gerar os filhos trocando as partes apos o ponto de corte
    filho1 = pai1[:ponto_corte] + pai2[ponto_corte:]
    filho2 = pai2[:ponto_corte] + pai1[ponto_corte:]

    return filho1, filho2

# Demonstracao
pai_a = [1, 1, 1, 1, 0, 0, 0, 0]
pai_b = [0, 0, 0, 0, 1, 1, 1, 1]
filho_a, filho_b = crossover(pai_a, pai_b)

print("Pai A:  ", pai_a)
print("Pai B:  ", pai_b)
print("Filho A:", filho_a)
print("Filho B:", filho_b)

---
## Passo 7: Mutacao (Inversao de Bit)

Para cada gene (bit) do cromossomo, ha uma **pequena chance** (5%) de ele ser invertido.  
Isso introduz **diversidade** e evita que o AG fique preso em otimos locais.

```
Antes:   [1, 0, 1, 0, 0, 1, 1, 0]
Mutacao no gene 4:
Depois:  [1, 0, 1, 0, 1, 1, 1, 0]
                      ^ invertido!
```

In [ ]:
def mutacao(individuo):
    """
    Aplica mutacao por inversao de bit em cada gene do cromossomo.
    Cada gene tem TAXA_MUTACAO de chance de ser invertido.
    """
    for i in range(NUM_ITENS):
        if random.random() < TAXA_MUTACAO:
            # Inverte o bit: 0 -> 1 ou 1 -> 0
            individuo[i] = 1 - individuo[i]
    return individuo

# Demonstracao com taxa de mutacao alta para visualizar
original = [0, 0, 0, 0, 0, 0, 0, 0]
print("Original:", original)
for i in range(5):
    copia = original[:]
    mutado = mutacao(copia)
    print("Mutado  :", mutado)

---
## Passo 8: Executar o Algoritmo Genetico!

Agora vamos juntar **tudo** no ciclo principal do AG:

```
1. Avaliar a populacao (calcular fitness de todos)
2. Selecionar pais (roleta)
3. Gerar filhos (crossover)
4. Aplicar mutacao nos filhos
5. Formar nova populacao
6. Repetir por N geracoes
```

Tambem usamos **Elitismo**: o melhor individuo passa direto para a proxima geracao, garantindo que nunca perdemos a melhor solucao.

In [ ]:
def algoritmo_genetico():
    """Executa o AG completo e retorna o melhor individuo encontrado."""

    # --- Criar populacao inicial aleatoria ---
    populacao = criar_populacao(TAMANHO_POPULACAO)

    # --- Guardar o melhor de todos os tempos ---
    melhor_global = None
    melhor_fitness_global = 0

    print("=" * 65)
    print("ALGORITMO GENETICO - PROBLEMA DA MOCHILA")
    print("Itens disponiveis:", NUM_ITENS, " |  Peso maximo:", PESO_MAXIMO, "kg")
    print("Populacao:", TAMANHO_POPULACAO, " |  Geracoes:", NUM_GERACOES)
    print("=" * 65)

    # --- Loop principal: evolucao por geracoes ---
    for geracao in range(NUM_GERACOES):

        # Passo 8.1: Calcular o fitness de toda a populacao
        fitness_lista = [calcular_fitness(ind) for ind in populacao]

        # Passo 8.2: Encontrar o melhor da geracao atual
        melhor_fitness = max(fitness_lista)
        indice_melhor = fitness_lista.index(melhor_fitness)
        melhor_individuo = populacao[indice_melhor]

        # Atualizar o melhor global (elitismo)
        if melhor_fitness > melhor_fitness_global:
            melhor_fitness_global = melhor_fitness
            melhor_global = melhor_individuo[:]

        # Exibir progresso a cada 10 geracoes e na primeira/ultima
        if geracao % 10 == 0 or geracao == NUM_GERACOES - 1:
            media_fitness = sum(fitness_lista) / len(fitness_lista)
            print("  Geracao", geracao, " |  Melhor: R$", round(melhor_fitness), " |  Media: R$", round(media_fitness))

        # Passo 8.3: Criar nova populacao
        nova_populacao = []

        # ELITISMO: o melhor individuo passa direto
        nova_populacao.append(melhor_individuo[:])

        # Gerar o restante da populacao
        while len(nova_populacao) < TAMANHO_POPULACAO:
            # Selecionar dois pais pela roleta
            pai1 = selecao_roleta(populacao, fitness_lista)
            pai2 = selecao_roleta(populacao, fitness_lista)

            # Aplicar crossover
            filho1, filho2 = crossover(pai1, pai2)

            # Aplicar mutacao
            filho1 = mutacao(filho1)
            filho2 = mutacao(filho2)

            # Adicionar filhos a nova populacao
            nova_populacao.append(filho1)
            if len(nova_populacao) < TAMANHO_POPULACAO:
                nova_populacao.append(filho2)

        # A nova geracao substitui a anterior
        populacao = nova_populacao

    return melhor_global, melhor_fitness_global

print("Funcao algoritmo_genetico() definida!")

---
## Passo 9: Rodar o AG e ver os resultados!

In [ ]:
# Rodar o AG
melhor_solucao, melhor_valor = algoritmo_genetico()

In [ ]:
# Mostrar a melhor solucao encontrada
print("=" * 65)
print("MELHOR SOLUCAO ENCONTRADA")
print("=" * 65)

peso_total = 0
valor_total = 0

print("\n  Item               Peso     Valor    Incluido?")
print("  ------------------ ------ --------   ---------")

for i in range(NUM_ITENS):
    incluido = "SIM <<<" if melhor_solucao[i] == 1 else "nao"
    print(" ", itens[i]["nome"], "\t", itens[i]["peso"], "kg  R$", itens[i]["valor"], " ", incluido)

    if melhor_solucao[i] == 1:
        peso_total += itens[i]["peso"]
        valor_total += itens[i]["valor"]

print("\n  Cromossomo:", melhor_solucao)
print("  Peso total: ", peso_total, "kg  (limite:", PESO_MAXIMO, "kg)")
print("  Valor total: R$", valor_total)
print("=" * 65)

---
## Passo 10: Comparacao com Forca Bruta

Para **validar** o AG, vamos testar **todas as combinacoes possiveis** (2^8 = 256) e encontrar a solucao otima.

O AG conseguiu encontrar a mesma solucao?

In [ ]:
# Forca bruta: testar todas as 2^N combinacoes possiveis
melhor_bruta_valor = 0
melhor_bruta_combo = None

for bits in range(2 ** NUM_ITENS):
    combo = [(bits >> i) & 1 for i in range(NUM_ITENS)]
    peso = sum(itens[i]["peso"] for i in range(NUM_ITENS) if combo[i] == 1)
    valor = sum(itens[i]["valor"] for i in range(NUM_ITENS) if combo[i] == 1)
    if peso <= PESO_MAXIMO and valor > melhor_bruta_valor:
        melhor_bruta_valor = valor
        melhor_bruta_combo = combo

print("COMPARACAO COM FORCA BRUTA")
print("Solucao otima:", melhor_bruta_combo, " ->  R$", melhor_bruta_valor)
print()

if melhor_valor == melhor_bruta_valor:
    print("O AG encontrou a solucao OTIMA!")
else:
    diferenca = (melhor_valor / melhor_bruta_valor) * 100
    print("O AG encontrou", round(diferenca, 1), "% do valor otimo.")
    print("(Rode novamente -- resultados podem variar por ser estocastico!)")

---
## Discussao em aula

### Exercicios propostos:

1. Mude o `PESO_MAXIMO` para 5.0 e rode novamente. A solucao muda?
2. Aumente a `TAXA_MUTACAO` para 0.5 (50%). O que acontece?
3. Reduza o `TAMANHO_POPULACAO` para 4. O AG ainda encontra a solucao otima?
4. Adicione mais itens a lista e observe o comportamento.

In [ ]:
# ESPACO PARA EXPERIMENTACAO DOS ALUNOS
# Mude os parametros acima e rode o AG novamente!

# Exemplo: mudando o peso maximo
# PESO_MAXIMO = 5.0
# melhor_solucao, melhor_valor = algoritmo_genetico()